# Week 8 - Notebook 4: Model Evaluation & Pipelines

**Course**: Introduction to Scientific Programming (CNC-UC)  
**Topic**: Cross-Validation, Pipelines, and Best Practices

---

## Learning Objectives

By the end of this notebook, you will:
- Understand cross-validation strategies
- Build scikit-learn pipelines for reproducible workflows
- Avoid common ML pitfalls (data leakage, overfitting)
- Perform hyperparameter tuning
- Create production-ready ML models
- Apply best practices to neuroscience research

## 1. Setup and Data Loading

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate,
    KFold, StratifiedKFold, GridSearchCV, RandomizedSearchCV,
    learning_curve, validation_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_squared_error, r2_score
)
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

# Configure plotting
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Libraries imported successfully")

## 2. Cross-Validation: The Gold Standard

**Problem with simple train/test split**: Results depend on which samples end up in each set.

**Solution**: **Cross-Validation** - Split data multiple times and average results.

### K-Fold Cross-Validation:
1. Split data into K equal parts (folds)
2. Train on K-1 folds, test on remaining fold
3. Repeat K times, rotating test fold
4. Average performance across all folds

**Benefits**:
- More reliable performance estimates
- Uses all data for both training and testing
- Reduces variance in estimates

In [ ]:
# Load breast cancer dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

print(f"Dataset: {cancer.target_names}")
print(f"Shape: {X.shape}")
print(f"Class balance: {np.bincount(y)}")

### 2.1 Compare Single Split vs Cross-Validation

In [ ]:
# Prepare model
model = LogisticRegression(max_iter=10000, random_state=42)

# Method 1: Single train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model.fit(X_train_scaled, y_train)
single_split_score = model.score(X_test_scaled, y_test)

print(f"Single Split Accuracy: {single_split_score:.4f}")

# Method 2: 5-Fold Cross-Validation
scaler_cv = StandardScaler()
X_scaled = scaler_cv.fit_transform(X)

cv_scores = cross_val_score(model, X_scaled, y, cv=5)

print(f"\n5-Fold Cross-Validation:")
print(f"  Fold scores: {cv_scores}")
print(f"  Mean accuracy: {cv_scores.mean():.4f}")
print(f"  Std deviation: {cv_scores.std():.4f}")
print(f"  95% CI: [{cv_scores.mean() - 1.96*cv_scores.std():.4f}, "
      f"{cv_scores.mean() + 1.96*cv_scores.std():.4f}]")

In [ ]:
# Visualize cross-validation scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot of fold scores
axes[0].bar(range(1, 6), cv_scores, color='steelblue', edgecolor='black')
axes[0].axhline(y=cv_scores.mean(), color='red', linestyle='--', 
               linewidth=2, label=f'Mean = {cv_scores.mean():.3f}')
axes[0].set_xlabel('Fold Number')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Cross-Validation Scores per Fold')
axes[0].set_ylim(0.9, 1.0)
axes[0].legend()

# Box plot comparing methods
comparison_data = [
    [single_split_score],
    cv_scores
]
axes[1].boxplot(comparison_data, tick_labels=['Single Split', '5-Fold CV'])
axes[1].scatter([1], [single_split_score], color='red', s=100, zorder=3, label='Single value')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Single Split vs Cross-Validation')
axes[1].set_ylim(0.9, 1.0)
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nKey Insight: CV gives us confidence intervals, not just a single number!")

### 2.2 Stratified K-Fold for Imbalanced Classes

**Stratified K-Fold** maintains class proportions in each fold - critical for imbalanced datasets.

In [ ]:
# Compare regular vs stratified K-Fold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Class distribution analysis:\n")
print(f"Overall: {np.bincount(y)} ({y.mean():.2%} positive class)\n")

print("Regular K-Fold splits:")
for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y), 1):
    train_dist = y[train_idx].mean()
    test_dist = y[test_idx].mean()
    print(f"  Fold {fold}: Train={train_dist:.2%}, Test={test_dist:.2%}")

print("\nStratified K-Fold splits:")
for fold, (train_idx, test_idx) in enumerate(stratified_kfold.split(X, y), 1):
    train_dist = y[train_idx].mean()
    test_dist = y[test_idx].mean()
    print(f"  Fold {fold}: Train={train_dist:.2%}, Test={test_dist:.2%}")

print("\n✓ Stratified K-Fold maintains consistent class proportions!")

## 3. Scikit-learn Pipelines: Production-Ready ML

**Pipelines** chain preprocessing and modeling steps.

**Why Use Pipelines?**
1. **Prevents data leakage**: Scaling fitted only on training data
2. **Cleaner code**: One object instead of multiple steps
3. **Easier deployment**: Save entire workflow as one object
4. **Cross-validation safe**: Preprocessing applied correctly in each fold

**Warning**: Fitting scaler on all data before cross-validation is a common mistake!

In [ ]:
# THE WRONG WAY (Data Leakage!)
print("❌ WRONG: Scaling before splitting\n")
X_scaled_wrong = StandardScaler().fit_transform(X)
cv_scores_wrong = cross_val_score(
    LogisticRegression(max_iter=10000), 
    X_scaled_wrong, y, cv=5
)
print(f"Mean CV Score (WRONG): {cv_scores_wrong.mean():.4f}")
print("Problem: Test data 'leaked' into scaling calculations!\n")

# THE RIGHT WAY (Using Pipeline)
print("✓ CORRECT: Using Pipeline\n")
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=10000))
])

cv_scores_correct = cross_val_score(pipeline, X, y, cv=5)
print(f"Mean CV Score (CORRECT): {cv_scores_correct.mean():.4f}")
print("Scaling fitted separately on each training fold!\n")

print(f"Difference: {cv_scores_wrong.mean() - cv_scores_correct.mean():.4f}")
print("The wrong way can give overly optimistic results!")

### 3.1 Build Complex Pipeline

In [ ]:
# Create pipeline with preprocessing, dimensionality reduction, and classification
complex_pipeline = Pipeline([
    ('scaler', StandardScaler()),              # Step 1: Scale features
    ('pca', PCA(n_components=10)),            # Step 2: Reduce dimensions
    ('classifier', RandomForestClassifier(    # Step 3: Classify
        n_estimators=100, 
        random_state=42
    ))
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train entire pipeline
complex_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = complex_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Complex Pipeline Performance:")
print(f"\nPipeline steps:")
for name, step in complex_pipeline.named_steps.items():
    print(f"  {name}: {type(step).__name__}")

print(f"\nTest Accuracy: {accuracy:.4f}")
print(f"\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

In [ ]:
# Access pipeline components
print("Accessing pipeline components:\n")

# Get PCA component
pca_component = complex_pipeline.named_steps['pca']
print(f"PCA explained variance: {pca_component.explained_variance_ratio_.sum():.2%}")

# Get classifier
classifier = complex_pipeline.named_steps['classifier']
print(f"Random Forest n_estimators: {classifier.n_estimators}")
print(f"Number of features after PCA: {pca_component.n_components_}")

## 4. Hyperparameter Tuning

**Hyperparameters**: Model settings that aren't learned from data (e.g., learning rate, tree depth).

### Grid Search vs Random Search

**Grid Search**:
- Try all combinations of specified parameter values
- Exhaustive but slow
- Good for small parameter spaces

**Random Search**:
- Try random combinations
- Faster, often finds good solutions
- Better for large parameter spaces

In [ ]:
# Define pipeline
tuning_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Define parameter grid
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [5, 10, 15, None],
    'classifier__min_samples_split': [2, 5, 10]
}

print(f"Grid Search Configuration:")
print(f"Parameters to search: {param_grid}")
total_combinations = 3 * 4 * 3
print(f"Total combinations: {total_combinations}")
print(f"With 5-fold CV: {total_combinations * 5} model fits\n")

# Perform grid search
print("Running Grid Search (this may take a minute)...")
grid_search = GridSearchCV(
    tuning_pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,  # Use all CPU cores
    verbose=0
)

grid_search.fit(X_train, y_train)

print("\n✓ Grid Search Complete")
print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")
print(f"Test Score: {grid_search.score(X_test, y_test):.4f}")

In [ ]:
# Visualize grid search results
results = pd.DataFrame(grid_search.cv_results_)

# Plot effect of n_estimators
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
for depth in results['param_classifier__max_depth'].unique():
    mask = results['param_classifier__max_depth'] == depth
    subset = results[mask].groupby('param_classifier__n_estimators')['mean_test_score'].mean()
    plt.plot(subset.index, subset.values, marker='o', label=f'max_depth={depth}')
plt.xlabel('n_estimators')
plt.ylabel('Mean CV Accuracy')
plt.title('Effect of n_estimators and max_depth')
plt.legend()
plt.grid(alpha=0.3)

# Plot top 10 configurations
plt.subplot(1, 2, 2)
top_10 = results.nsmallest(10, 'rank_test_score')
plt.barh(range(10), top_10['mean_test_score'].values)
plt.yticks(range(10), [f"Config {i+1}" for i in range(10)])
plt.xlabel('Mean CV Accuracy')
plt.title('Top 10 Configurations')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

### 4.1 Random Search: Faster Alternative

In [ ]:
# Define parameter distributions for random search
from scipy.stats import randint, uniform

param_distributions = {
    'classifier__n_estimators': randint(50, 200),
    'classifier__max_depth': [5, 10, 15, 20, None],
    'classifier__min_samples_split': randint(2, 20),
    'classifier__min_samples_leaf': randint(1, 10)
}

print("Random Search Configuration:")
print(f"Will try 20 random combinations (vs {total_combinations} in grid search)\n")

# Perform random search
print("Running Random Search...")
random_search = RandomizedSearchCV(
    tuning_pipeline,
    param_distributions,
    n_iter=20,  # Try 20 random combinations
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=0
)

random_search.fit(X_train, y_train)

print("\n✓ Random Search Complete")
print(f"\nBest Parameters: {random_search.best_params_}")
print(f"Best CV Score: {random_search.best_score_:.4f}")
print(f"Test Score: {random_search.score(X_test, y_test):.4f}")

print(f"\nComparison:")
print(f"Grid Search best CV: {grid_search.best_score_:.4f}")
print(f"Random Search best CV: {random_search.best_score_:.4f}")
print(f"\nRandom Search explored {20/total_combinations:.1%} of parameter space!")

## 5. Diagnosing Model Performance

### Learning Curves: Detecting Overfitting/Underfitting

In [ ]:
# Create learning curves
simple_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=10000))
])

train_sizes, train_scores, val_scores = learning_curve(
    simple_pipeline, X, y,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Calculate means and stds
train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Training score')
plt.fill_between(train_sizes, 
                 train_mean - train_std, 
                 train_mean + train_std,
                 alpha=0.2, color='blue')

plt.plot(train_sizes, val_mean, 'o-', color='red', label='Validation score')
plt.fill_between(train_sizes, 
                 val_mean - val_std, 
                 val_mean + val_std,
                 alpha=0.2, color='red')

plt.xlabel('Training Set Size')
plt.ylabel('Accuracy')
plt.title('Learning Curves', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

print("Learning Curve Interpretation:")
print("- Training and validation curves converge: Good fit")
print("- Large gap between curves: Overfitting")
print("- Both curves plateau at low score: Underfitting")
print("- Curves still improving: More data might help")

### Validation Curves: Tuning Single Parameter

In [ ]:
# Create validation curve for regularization parameter
C_range = np.logspace(-3, 3, 10)

train_scores_vc, val_scores_vc = validation_curve(
    Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=10000))
    ]),
    X, y,
    param_name='classifier__C',
    param_range=C_range,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

train_mean_vc = train_scores_vc.mean(axis=1)
train_std_vc = train_scores_vc.std(axis=1)
val_mean_vc = val_scores_vc.mean(axis=1)
val_std_vc = val_scores_vc.std(axis=1)

plt.figure(figsize=(10, 6))
plt.semilogx(C_range, train_mean_vc, 'o-', color='blue', label='Training score')
plt.fill_between(C_range, 
                 train_mean_vc - train_std_vc,
                 train_mean_vc + train_std_vc,
                 alpha=0.2, color='blue')

plt.semilogx(C_range, val_mean_vc, 'o-', color='red', label='Validation score')
plt.fill_between(C_range, 
                 val_mean_vc - val_std_vc,
                 val_mean_vc + val_std_vc,
                 alpha=0.2, color='red')

plt.xlabel('Regularization Parameter C')
plt.ylabel('Accuracy')
plt.title('Validation Curve: Effect of Regularization', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

optimal_C = C_range[np.argmax(val_mean_vc)]
print(f"\nOptimal C value: {optimal_C:.3f}")
print(f"Validation accuracy at optimal C: {val_mean_vc.max():.4f}")

## 6. Common Pitfalls and Solutions

### Pitfall 1: Data Leakage

In [ ]:
print("=" * 60)
print("COMMON PITFALL #1: DATA LEAKAGE")
print("=" * 60)

# Load diabetes dataset for regression example
diabetes = load_diabetes()
X_diab = diabetes.data
y_diab = diabetes.target

# WRONG: Fit scaler on entire dataset
scaler_wrong = StandardScaler()
X_scaled_all = scaler_wrong.fit_transform(X_diab)
X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_scaled_all, y_diab, test_size=0.2, random_state=42
)
model_leak = Ridge()
model_leak.fit(X_train_leak, y_train_leak)
r2_leak = model_leak.score(X_test_leak, y_test_leak)

# CORRECT: Fit scaler only on training data
X_train_correct, X_test_correct, y_train_correct, y_test_correct = train_test_split(
    X_diab, y_diab, test_size=0.2, random_state=42
)
scaler_correct = StandardScaler()
X_train_scaled_correct = scaler_correct.fit_transform(X_train_correct)
X_test_scaled_correct = scaler_correct.transform(X_test_correct)
model_correct = Ridge()
model_correct.fit(X_train_scaled_correct, y_train_correct)
r2_correct = model_correct.score(X_test_scaled_correct, y_test_correct)

print(f"\n❌ WITH DATA LEAKAGE:   R² = {r2_leak:.4f}")
print(f"✓  WITHOUT DATA LEAKAGE: R² = {r2_correct:.4f}")
print(f"\nDifference: {r2_leak - r2_correct:.4f}")
print("\nSolution: Always use Pipelines or fit preprocessing only on training data!")

### Pitfall 2: Test Set Abuse

In [ ]:
print("=" * 60)
print("COMMON PITFALL #2: TEST SET ABUSE")
print("=" * 60)

print("\n❌ WRONG: Using test set to tune hyperparameters")
print("   1. Try parameter A → test on test set → score = 0.90")
print("   2. Try parameter B → test on test set → score = 0.92")
print("   3. Try parameter C → test on test set → score = 0.91")
print("   4. Choose B because it has highest test score")
print("   Problem: Test set performance is now overly optimistic!")

print("\n✓ CORRECT: Three-way split")
print("   1. Split: 60% train, 20% validation, 20% test")
print("   2. Tune on validation set")
print("   3. Evaluate final model on test set (only once!)")

# Demonstrate correct approach
X_temp, X_test_final, y_temp, y_test_final = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"\nDataset sizes:")
print(f"  Training:   {len(X_train_final)} ({len(X_train_final)/len(X)*100:.0f}%)")
print(f"  Validation: {len(X_val)} ({len(X_val)/len(X)*100:.0f}%)")
print(f"  Test:       {len(X_test_final)} ({len(X_test_final)/len(X)*100:.0f}%)")

print("\nRemember: Test set is sacred - use it only for final evaluation!")

## 7. Complete Workflow Example

Putting it all together: A production-ready ML workflow.

In [ ]:
print("=" * 70)
print("COMPLETE PRODUCTION-READY ML WORKFLOW")
print("=" * 70)

# Step 1: Load data
print("\n[1/6] Loading data...")
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
print(f"✓ Loaded {X.shape[0]} samples with {X.shape[1]} features")

# Step 2: Split data
print("\n[2/6] Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"✓ Train: {len(X_train)}, Test: {len(X_test)}")

# Step 3: Define pipeline
print("\n[3/6] Creating pipeline...")
production_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95)),  # Keep 95% variance
    ('classifier', RandomForestClassifier(random_state=42))
])
print("✓ Pipeline: Scaler → PCA → RandomForest")

# Step 4: Hyperparameter tuning
print("\n[4/6] Tuning hyperparameters...")
param_grid_prod = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, 20, None],
    'classifier__min_samples_split': [2, 5]
}

grid_search_prod = GridSearchCV(
    production_pipeline,
    param_grid_prod,
    cv=5,
    scoring='f1',  # Use F1 for medical diagnosis
    n_jobs=-1,
    verbose=0
)
grid_search_prod.fit(X_train, y_train)
print(f"✓ Best CV F1: {grid_search_prod.best_score_:.4f}")
print(f"✓ Best params: {grid_search_prod.best_params_}")

# Step 5: Evaluate on test set
print("\n[5/6] Evaluating on test set...")
y_pred_prod = grid_search_prod.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred_prod)
test_precision = precision_score(y_test, y_pred_prod)
test_recall = recall_score(y_test, y_pred_prod)
test_f1 = f1_score(y_test, y_pred_prod)

print(f"✓ Test Accuracy:  {test_accuracy:.4f}")
print(f"✓ Test Precision: {test_precision:.4f}")
print(f"✓ Test Recall:    {test_recall:.4f}")
print(f"✓ Test F1:        {test_f1:.4f}")

# Step 6: Final model ready for deployment
print("\n[6/6] Model ready for deployment")
print(f"✓ Final pipeline trained on {len(X_train)} samples")
print(f"✓ PCA retained {grid_search_prod.best_estimator_.named_steps['pca'].n_components_} components")
print("✓ Model can be saved with joblib.dump()")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_prod)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=cancer.target_names,
           yticklabels=cancer.target_names)
plt.title('Final Model: Confusion Matrix', fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print("\n" + "=" * 70)
print("WORKFLOW COMPLETE - MODEL READY FOR PRODUCTION")
print("=" * 70)

## 8. Key Takeaways

### Best Practices Checklist:

✓ **Always use cross-validation** for reliable performance estimates

✓ **Use stratified splits** for classification with imbalanced classes

✓ **Build pipelines** to prevent data leakage

✓ **Tune hyperparameters** on validation set, not test set

✓ **Use test set only once** for final evaluation

✓ **Check learning curves** to diagnose overfitting/underfitting

✓ **Choose metrics carefully** based on your problem (accuracy, F1, recall, etc.)

✓ **Scale features** for distance-based algorithms (SVM, KNN)

✓ **Set random seeds** for reproducibility

✓ **Document your workflow** for scientific reproducibility

### For Research:

- **Small sample sizes**: Cross-validation is crucial
- **Imbalanced classes**: Use stratified splits and appropriate metrics
- **High-dimensional data**: Consider dimensionality reduction in pipeline
- **Reproducibility**: Always set random seeds and document preprocessing
- **Interpretability**: Balance performance with explainability for scientific insights

## 9. Practice Exercises

1. **Three-Way Split**: Implement train/validation/test split and tune a model using only validation set.

2. **Pipeline Comparison**: Create two pipelines (with and without PCA) and compare using cross-validation.

3. **Learning Curves**: Generate learning curves for different model complexities. Which shows overfitting?

4. **Hyperparameter Tuning**: Use RandomizedSearchCV with 50 iterations. How does it compare to GridSearchCV?

5. **Your Research**: Apply these best practices to your data. Build a production-ready pipeline!